# iSAGE — Iterative Sparse Annotation Guided by Expert

A 3-cell workflow:

1. **Setup** — imports.
2. **Configure session** — pick dataset + training config + session name, click *Setup / Load*.
3. **Loop**: annotate → train → repeat.

Each iteration of the loop runs two more cells (annotate, then train). The
session manager remembers where you are: re-running the loop cells always
picks up at the latest iteration.


In [ ]:
# Cell 1 — Setup

import sys
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import imageio
from tqdm import tqdm
import ipywidgets as W
from IPython.display import display, clear_output

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Project modules
from src.utils.config_loader import load_dataset_config, load_training_config
from src.session.manager import get_or_create_session
from src.session.mask_utils import batch_json_to_masks
from src.session.simple_mask_converter import convert_mask_to_json
from src.training.setup import setup_training
from src.training.workflow import run_training_iteration
from src.annotation.launcher import run_annotation_workflow

# Torch
import torch
print(f"PyTorch {torch.__version__}, CUDA={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

import segmentation_models_pytorch as smp
print(f"smp from: {smp.__file__}")

%matplotlib inline


## Cell 2 — Configure session

Pick a dataset YAML, a training config, and a session name. Click **Setup / Load**.

- The dataset YAML lives in `configs/datasets/`. New dataset? Drop a YAML there and refresh the cell.
- The training YAML lives in `configs/training/`. `unet_efficientnet_b7.yaml` is the paper default (EWDLMulticlass, λ=5).
- The session name is either an existing one (loads where you left off) or a new one (created fresh).


In [ ]:
# Cell 2 — Session manager widget

_datasets_dir = Path('configs/datasets')
_trainings_dir = Path('configs/training')
_sessions_dir = Path('Sessions')

_dataset_options = sorted(f.name for f in _datasets_dir.glob('*.yaml'))
_training_options = sorted(f.name for f in _trainings_dir.glob('*.yaml'))
_existing_sessions = sorted(d.name for d in _sessions_dir.iterdir() if d.is_dir()) if _sessions_dir.exists() else []

_default_training = 'unet_efficientnet_b7.yaml' if 'unet_efficientnet_b7.yaml' in _training_options else (_training_options[0] if _training_options else None)
_default_dataset = _dataset_options[0] if _dataset_options else None
_default_session = _existing_sessions[0] if _existing_sessions else 'my_run'

_dataset_w  = W.Dropdown(options=_dataset_options, value=_default_dataset, description='Dataset:', layout=W.Layout(width='600px'), style={'description_width': '100px'})
_training_w = W.Dropdown(options=_training_options, value=_default_training, description='Training:', layout=W.Layout(width='600px'), style={'description_width': '100px'})
_session_w  = W.Combobox(options=_existing_sessions, value=_default_session, placeholder='session name', description='Session:', layout=W.Layout(width='600px'), style={'description_width': '100px'}, ensure_option=False)
_btn        = W.Button(description='Setup / Load', button_style='primary', icon='play', layout=W.Layout(width='200px'))
_out        = W.Output()

def _setup_and_load(_):
    global dataset_config, training_config, model, device, train_loss, val_loss, metrics, optimizer, SESSION_PATH, session_info
    with _out:
        clear_output()
        try:
            dataset_config  = load_dataset_config(f'configs/datasets/{_dataset_w.value}')
            training_config = load_training_config(f'configs/training/{_training_w.value}')
            print(f"Dataset:  {dataset_config['name']} — {dataset_config['classes']['num_classes']} classes ({', '.join(dataset_config['classes']['names'])})")
            print(f"Training: {training_config['name']} — {training_config['model']['architecture']} + {training_config['model']['encoder']}")
            print()
            model, device, train_loss, val_loss, metrics, optimizer = setup_training(
                dataset_config=dataset_config,
                training_config=training_config,
            )
            print()
            SESSION_PATH = Path(f'Sessions/{_session_w.value}')
            session_info = get_or_create_session(
                session_path=SESSION_PATH,
                dataset_config=dataset_config,
            )
            print(f"\n→ Ready. Run the annotate cell next.")
        except Exception as e:
            print(f"FAILED: {type(e).__name__}: {e}")
            import traceback
            traceback.print_exc()

_btn.on_click(_setup_and_load)
display(W.VBox([_dataset_w, _training_w, _session_w, _btn, _out]))


## Cell 3 — Annotate

Launches the PyQt5 annotation widget on the latest iteration. Left-click adds a point for the current class; right-click removes a nearby point. Close the window to advance.

On close: JSONs auto-save and PNG masks regenerate.


In [ ]:
# Cell 3 — Annotate

result = run_annotation_workflow(
    session_path=SESSION_PATH,
    dataset_config=dataset_config,
    iteration='latest',
    launch_tool=True,
)


## Cell 4 — Train

Trains the current model on the latest iteration's annotations using EWDL,
then generates predictions and advances to the next iteration. Plots the
mIoU progression below.


In [ ]:
# Cell 4 — Train

result = run_training_iteration(
    session_path=SESSION_PATH,
    dataset_config=dataset_config,
    training_config=training_config,
    model=model,
    device=device,
    train_loss=train_loss,
    val_loss=val_loss,
    metrics=metrics,
    optimizer=optimizer,
    iteration='latest',
    visualize=False,
)

# Plot mIoU history across iterations
_csv = SESSION_PATH / 'metrics_history.csv'
if _csv.exists():
    _h = pd.read_csv(_csv)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(_h['iteration'], _h['miou'], marker='o', linewidth=2, color='#1f77b4', label='val mIoU')
    if 'pixel_accuracy' in _h.columns:
        ax.plot(_h['iteration'], _h['pixel_accuracy'], marker='s', linewidth=1.5, color='#7f7f7f', alpha=0.6, label='pixel acc')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Metric')
    ax.set_title(f'{SESSION_PATH.name} — progression')
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print('metrics_history.csv not found yet (first training will create it).')
